# Argentina Solar Station Processing Tutorial

This notebook processes Argentinian station observations and creates the monthly station climatology used by the ATLAS solar workflow.

The final output is:

```text
../data/stations/argentina/allstats_solar_radiation.csv
```

Users should only edit the **User parameters** cell, then run the notebook from top to bottom.

## Step 1. User parameters

The input folder must contain one subfolder per station:

```text
../data/stations/argentina/{station_name}/
```

Each station subfolder may contain one or more Excel files.

In [6]:
from pathlib import Path

country = "argentina"

# Main input and output folder.
base_path = Path(f"../data/stations/{country}")
output_path = base_path
output_path.mkdir(parents=True, exist_ok=True)

# Final output.
output_csv = output_path / "allstats_solar_radiation.csv"

# Stations to process.
station_names = [
    "Ushuaia",
    "Tucumán",
    "Rio_Gallegos",
    "Pilar",
    "Neuquén",
    "Mendoza",
    "La_Quiaca",
    "Comodoro_Rivadavia",
    "Buenos_Aires",
    "Bariloche",
]

# Station coordinates: [latitude, longitude].
station_coordinates = {
    "Buenos_Aires": [-34.590203, -58.483655],
    "Pilar": [-31.6681, -63.882],
    "La_Quiaca": [-22.1038, -65.60125],
    "Tucumán": [-26.83377, -65.1068],
    "Mendoza": [-32.894759, -68.872825],
    "Bariloche": [-41.14884, -71.1628],
    "Neuquén": [-38.9516, -68.1474],
    "Rio_Gallegos": [-51.612078, -69.305858],
    "Ushuaia": [-54.847713, -68.307876],
    "Comodoro_Rivadavia": [-45.78784, -67.468844],
}

# Optional station IDs. Leave empty if no official codes are available.
station_ids = {}

# Column keywords used to detect radiation measurements.
radiation_keywords = [
    "IRRAD (W/M2)",
    "IRRADIANCIA(W/M2)",
    "GHI_W_M2",
    "RAD",
    "IRRADIANCIA (W/M2)",
    "GLOBAL",
]

## Step 2. Imports

In [7]:
import glob
import os
import re

import numpy as np
import pandas as pd

## Step 3. Helper functions

In [8]:
def normalise_columns(df):
    """Standardise column names and remove empty columns.

    This function also converts Spanish characters, so columns such as
    AÑO become ANO and can be handled consistently.
    """
    df = df.copy()

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace("Á", "A", regex=False)
        .str.replace("É", "E", regex=False)
        .str.replace("Í", "I", regex=False)
        .str.replace("Ó", "O", regex=False)
        .str.replace("Ú", "U", regex=False)
        .str.replace("Ñ", "N", regex=False)
    )

    df = df.loc[:, ~df.columns.str.startswith("UNNAMED")]
    df = df.dropna(axis=1, how="all")

    return df


def extract_year_from_filename(filepath):
    """Extract the first year found in a file name."""
    filename = os.path.basename(filepath)
    match = re.search(r"(19|20)\d{2}", filename)
    return int(match.group()) if match else None


def read_excel_flexible(filepath):
    """Read an Excel file and return a table with a detected header.

    Some station files have a regular first-row header, while others may
    contain metadata rows before the actual table. This function first tries
    the standard read. If expected columns are not found, it scans the first
    rows and uses the row containing the station table header.
    """
    filepath = Path(filepath)

    # First attempt: standard Excel layout with header in the first row.
    df = pd.read_excel(filepath)
    df_norm = normalise_columns(df)

    expected_any = {"ANO", "FECHA", "FECHA_HORA", "FECHA/HORA", "DATETIME", "DIA", "DIA JULIANO"}
    if expected_any.intersection(set(df_norm.columns)):
        return df_norm

    # Second attempt: scan without headers and look for the actual table header.
    raw = pd.read_excel(filepath, header=None)

    for idx in range(min(30, len(raw))):
        row_values = (
            raw.iloc[idx]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.replace("Ñ", "N", regex=False)
            .tolist()
        )

        if ("ANO" in row_values or "AÑO" in row_values) and ("DIA" in row_values or "DIA JULIANO" in row_values):
            header = raw.iloc[idx].astype(str).str.strip().tolist()
            df = raw.iloc[idx + 1 :].copy()
            df.columns = header
            return normalise_columns(df)

        if "FECHA" in row_values or "FECHA_HORA" in row_values or "FECHA/HORA" in row_values:
            header = raw.iloc[idx].astype(str).str.strip().tolist()
            df = raw.iloc[idx + 1 :].copy()
            df.columns = header
            return normalise_columns(df)

    # Fallback: return the standard version.
    return df_norm


def parse_hora_column(values):
    """Parse the HORA column.

    Supported formats:
    - 1 to 24: hourly values
    - 0 to 23: hourly values
    - HHMM format, for example 1, 15, 100, 930, 2359, 2400

    For HHMM format, 2400 is converted to 00:00 of the following day.
    The function returns two Series: minutes within the day and day offset.
    """
    h = pd.to_numeric(values, errors="coerce")

    # If values larger than 59 exist, interpret the column as HHMM.
    # This is the format used by files such as "RAD GLOB MDZ 2016.xlsx".
    if h.max(skipna=True) > 59:
        h_int = h.round().astype("Int64")
        hours = (h_int // 100).astype("Float64")
        minutes = (h_int % 100).astype("Float64")

        day_offset = (hours == 24).astype(int)
        hours = hours.where(hours < 24, 0)

        invalid = (minutes >= 60) | (hours > 24) | (hours < 0)
        hours = hours.mask(invalid)
        minutes = minutes.mask(invalid)

        total_minutes = hours * 60 + minutes
        return total_minutes, day_offset

    # If values are 1 to 24, interpret them as hourly observations ending at that hour.
    # Hour 24 becomes 00:00 of the following day.
    if h.min(skipna=True) >= 1 and h.max(skipna=True) <= 24:
        hours = h.round().astype("Int64")
        day_offset = (hours == 24).astype(int)
        hours = hours.where(hours < 24, 0)
        total_minutes = hours.astype("Float64") * 60
        return total_minutes, day_offset

    # If values are 0 to 23, interpret them as standard hours.
    hours = h.round().astype("Float64")
    day_offset = pd.Series(0, index=values.index)
    total_minutes = hours * 60
    return total_minutes, day_offset


def preprocess_time(df, year=None):
    """Create a common time column from different station file formats."""
    df = normalise_columns(df)

    for datetime_col in ["FECHA_HORA", "FECHA/HORA", "DATETIME", "DATE_TIME"]:
        if datetime_col in df.columns:
            df["time"] = pd.to_datetime(df[datetime_col], errors="coerce")
            return df

    if "FECHA" in df.columns and "HORA" in df.columns and "MINUTO" in df.columns:
        fecha = pd.to_datetime(df["FECHA"], errors="coerce")
        hora = pd.to_numeric(df["HORA"], errors="coerce").fillna(0)
        minuto = pd.to_numeric(df["MINUTO"], errors="coerce").fillna(0)
        df["time"] = fecha + pd.to_timedelta(hora, unit="h") + pd.to_timedelta(minuto, unit="m")
        return df

    if "FECHA" in df.columns:
        fecha = pd.to_datetime(df["FECHA"], errors="coerce")
        if fecha.notna().sum() > 0:
            df["time"] = fecha
            return df

    if "ANO" in df.columns:
        years = pd.to_numeric(df["ANO"], errors="coerce")
    elif year is not None:
        years = pd.Series(year, index=df.index)
    else:
        raise ValueError("A year column or a year in the file name is required.")

    julian_col = None
    if "DIA JULIANO" in df.columns:
        julian_col = "DIA JULIANO"
    elif "DIA" in df.columns:
        julian_col = "DIA"

    if julian_col is not None and "HORA" in df.columns:
        julian = pd.to_numeric(df[julian_col], errors="coerce")

        base = pd.to_datetime(
            years.round().astype("Int64").astype(str),
            format="%Y",
            errors="coerce",
        )

        total_minutes, day_offset = parse_hora_column(df["HORA"])

        df["time"] = (
            base
            + pd.to_timedelta(julian - 1, unit="D")
            + pd.to_timedelta(total_minutes, unit="m")
            + pd.to_timedelta(day_offset, unit="D")
        )

        return df

    raise ValueError("Unable to create a time column from this file.")


def find_radiation_columns(columns, radiation_keywords):
    """Find possible radiation columns using a list of keywords."""
    return [
        col
        for col in columns
        if any(keyword.upper() in str(col).upper() for keyword in radiation_keywords)
    ]


def standardise_output(df):
    """Return the common ATLAS station table layout."""
    df = df.copy()
    df["id"] = df["name"].map(station_ids)
    df["latitude"] = df["name"].map(lambda x: station_coordinates.get(x, [np.nan, np.nan])[0])
    df["longitude"] = df["name"].map(lambda x: station_coordinates.get(x, [np.nan, np.nan])[1])
    df["altitude"] = np.nan

    expected_columns = ["id", "month", "ssrd", "name", "latitude", "longitude", "altitude"]
    for col in expected_columns:
        if col not in df.columns:
            df[col] = pd.NA

    return df.loc[:, expected_columns].sort_values(["name", "month"]).reset_index(drop=True)

## Step 4. Processing functions

In [9]:
def read_station_folder(station_name, base_path):
    """Read and merge all Excel files for one station."""
    station_folder = Path(base_path) / station_name
    files = sorted(list(station_folder.glob("*.xlsx")) + list(station_folder.glob("*.xls")))

    if not files:
        print(f"No Excel files found for {station_name}: {station_folder}")
        return pd.DataFrame()

    station_tables = []

    for file in files:
        try:
            year = extract_year_from_filename(file)
            df = read_excel_flexible(file)
            df = preprocess_time(df, year=year)
            df = df.dropna(subset=["time"])
            df["name"] = station_name
            station_tables.append(df)
        except Exception as exc:
            print(f"Skipping {file.name}: {exc}")

    if not station_tables:
        return pd.DataFrame()

    return pd.concat(station_tables, ignore_index=True).sort_values("time")


def station_monthly_climatology(df_station):
    """Create a monthly climatology for one station."""
    df = df_station.copy()
    df.columns = df.columns.str.lower()

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["ssrd"] = pd.to_numeric(df["ssrd"], errors="coerce")

    station_name = df["name"].iloc[0] if "name" in df.columns and len(df) else "unknown"

    df = df.dropna(subset=["time", "ssrd"])
    df = df[df["ssrd"] >= 0]

    if df.empty:
        return pd.DataFrame({"month": range(1, 13), "ssrd": np.nan, "name": station_name})

    q1 = df["ssrd"].quantile(0.25)
    q3 = df["ssrd"].quantile(0.75)
    iqr = q3 - q1

    if pd.notna(iqr) and iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df = df[(df["ssrd"] >= lower) & (df["ssrd"] <= upper)]

    if df.empty:
        return pd.DataFrame({"month": range(1, 13), "ssrd": np.nan, "name": station_name})

    df = df.set_index("time").sort_index()
    daily = df[["ssrd"]].resample("1D").mean()

    climatology = (
        daily.groupby(daily.index.month)
        .mean(numeric_only=True)
        .reindex(range(1, 13))
    )

    climatology.index.name = "month"
    climatology = climatology.reset_index()
    climatology["name"] = station_name

    return climatology


def process_all_stations(base_path, station_names, radiation_keywords):
    """Read all stations and create one monthly climatology table."""
    all_tables = []

    for station_name in station_names:
        df_station = read_station_folder(station_name, base_path)
        if df_station.empty:
            continue

        radiation_columns = find_radiation_columns(df_station.columns, radiation_keywords)
        if not radiation_columns:
            print(f"Skipping {station_name}: no radiation column found. Available columns: {list(df_station.columns)}")
            continue

        df_station = df_station.copy()
        df_station["ssrd"] = df_station[radiation_columns].bfill(axis=1).iloc[:, 0]
        df_station = df_station.dropna(subset=["name", "ssrd", "time"])
        df_station = df_station[["time", "name", "ssrd"]]

        all_tables.append(df_station)

    if not all_tables:
        raise ValueError("No station data could be processed.")

    merged = pd.concat(all_tables, ignore_index=True).sort_values("name")

    climatology = (
        merged.groupby("name", group_keys=False)
        .apply(station_monthly_climatology)
        .reset_index(drop=True)
    )

    return standardise_output(climatology)

## Step 5. Run processing

In [10]:
df_climatology_all = process_all_stations(
    base_path=base_path,
    station_names=station_names,
    radiation_keywords=radiation_keywords,
)

df_climatology_all.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows: {len(df_climatology_all)}")
print(f"Stations: {df_climatology_all['name'].nunique()}")

/tmp/ipykernel_1409059/2775098387.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(station_monthly_climatology)


Saved: /mnt/DATA/PROGETTI/27_WMO_ATLAS/stations/argentina/allstats_solar_radiation.csv
Rows: 120
Stations: 10


## Step 6. Quick quality check

In [11]:
df_climatology_all.head()

,id,month,ssrd,name,latitude,longitude,altitude
0,NaN,1,584.825692,Bariloche,-41.14884,-71.1628,NaN
1,NaN,2,518.507554,Bariloche,-41.14884,-71.1628,NaN
2,NaN,3,443.787475,Bariloche,-41.14884,-71.1628,NaN
3,NaN,4,303.192675,Bariloche,-41.14884,-71.1628,NaN
4,NaN,5,187.631942,Bariloche,-41.14884,-71.1628,NaN
